In [3]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

nombre_carpeta = "deteccion_defectos-1"

# os.path.isdir verifica la existencia y que sea una carpeta al mismo tiempo
if os.path.isdir(nombre_carpeta):
    print(f"¡La carpeta '{nombre_carpeta}' existe!")
else:
    print(f"La carpeta '{nombre_carpeta}' no existe, se descargará de roboflow.")
    # Carga las variables del archivo .env
    load_dotenv()
    
    # Obtiene la API Key de las variables de entorno
    ROBOFLOW_KEY = os.getenv("API_ROBOFLOW")
    ROBOFLOW_WORKSPACE = os.getenv("API_ROBOFLOW_WORKSPACE")
    
    if not ROBOFLOW_KEY:
        raise ValueError("No se encontró la variable de entorno 'API_ROBOFLOW'. Asegúrate de definirla en tu archivo .env")
    rf = Roboflow(api_key=f"{ROBOFLOW_KEY}")
    project = rf.workspace("clothesdataset-yimbv").project("deteccion_defectos")
    version = project.version(1)
    dataset = version.download("yolov11")
    print(f"Dataset descargado de roboflow")

La carpeta 'deteccion_defectos-1' no existe, se descargará de roboflow.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to deteccion_defectos-1 in yolov11:: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 273/273 [00:00<00:00, 2288.78it/s]

Dataset descargado de roboflow


In [6]:
import yaml
import os

# Define las rutas absolutas o relativas desde donde ejecutarás el entrenamiento
# Es mejor usar rutas completas para evitar errores de "File Not Found"
dataset_path = os.path.abspath(f"{nombre_carpeta}")

data_config = {
    'path': dataset_path,      # Directorio raíz del dataset
    'train': 'train/images',   # Ruta relativa a 'path' para entrenamiento
    'val': 'valid/images',     # Ruta relativa a 'path' para validación
    'test': 'test/images',     # Ruta relativa a 'path' para pruebas (opcional)

    'nc': 2,                   # Número de clases
    'names': ['Corrido', 'Hueco'] # Asegúrate de que este orden sea el mismo de Roboflow
}

# Guardar el archivo
with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Archivo data.yaml creado con éxito.")

Archivo data.yaml creado con éxito.


In [10]:
import torch
from ultralytics import YOLO
import os
import gc

def clear_gpu():
    torch.cuda.empty_cache()
    gc.collect()
    # Forzamos a que PyTorch no sea tan rígido con la memoria en la serie 50
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if __name__ == '__main__':
    clear_gpu()

    # Cargamos la versión 11 Medium
    model = YOLO('yolo11m.pt') 
    
    # Entrenamiento
    model.train(
        data="data.yaml",
        epochs=150,          # Al ser un dataset pequeño, dale tiempo a converger
        imgsz=1280,          # Tu resolución objetivo para conservar detalles pequeños
        batch=2,             # Bajamos a 2 para que quepa cómodamente en tus 8GB de VRAM
        amp=True,            # Fuerza precisión mixta (FP16) reduciendo el consumo a la mitad
        workers=2,           # Bajamos los workers para no saturar la transferencia CPU-GPU
        
        # --- Palancas de Mosaico y Aumentación ---
        mosaic=1.0,          # Activa al 100% la aumentación en mosaico de 4 imágenes
        mixup=0.15,          # Mezcla dos imágenes superpuestas (ayuda con texturas textiles)
        copy_paste=0.3,      # Clave: Pega defectos de una foto en otra para balancear 'hueco'
        
        # --- Palancas de Balanceo y Enfoque ---
        box=7.5,             # Le da prioridad milimétrica a la precisión de las cajas
        cls=1.5,             # Compensa el desbalance dándole 150% de peso a la clasificación
        
        # --- Variación de Color y Entorno ---
        hsv_h=0.15,          # Varía el tono para cubrir la diversidad de hilos/colores
        hsv_s=0.7,           # Varía la saturación
        hsv_v=0.4,           # Varía el brillo para emular sombras en el tejido

        # --- HARDWARE ---
        cache=False,
        device=0,
        name='defectos_v11m'
    )

New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.15, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=defectos_v11m, nbs=64, nms=False, opset=None,

In [13]:
metrics = model.val(conf=0.3)  # Filtra detecciones dudosas para maximizar precisión
print(metrics.results_dict['metrics/precision(B)'])

Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 1952.2552.0 MB/s, size: 550.6 KB)
val: Scanning C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\labels.cache... 20 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.5s/it 3.0s9.1s
                   all         20        381      0.824      0.546      0.693      0.323
               Corrido         13        295      0.803      0.651      0.749      0.381
                 Hueco         13         86      0.844      0.442      0.637      0.265
Speed: 7.6ms preprocess, 21.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\val4
0.8238958623895862


# Prueba

In [14]:
import cv2
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

def tile_dataset(img_dir, label_dir, output_img_dir, output_label_dir, tile_size=1280, overlap=0.2):
    Path(output_img_dir).mkdir(parents=True, exist_ok=True)
    Path(output_label_dir).mkdir(parents=True, exist_ok=True)
    
    img_files = list(Path(img_dir).glob("*.jpg"))
    stride = int(tile_size * (1 - overlap)) # Desplazamiento considerando el solape

    for img_path in tqdm(img_files, desc="Procesando Tiling"):
        # 1. Cargar imagen original 4K
        img = cv2.imread(str(img_path))
        h, w, _ = img.shape
        
        # 2. Cargar etiquetas originales (formato YOLO: class x_cen y_cen width height)
        label_path = Path(label_dir) / f"{img_path.stem}.txt"
        bboxes = []
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 5:
                        bboxes.append([int(parts[0])] + [float(x) for x in parts[1:]])

        # 3. Recorrer la imagen en una cuadrícula (Grid)
        tile_count = 0
        for y in range(0, h - tile_size + 1, stride):
            for x in range(0, w - tile_size + 1, stride):
                
                # Coordenadas absolutas del parche actual
                x1, y1 = x, y
                x2, y2 = x + tile_size, y + tile_size
                
                # Extraer el parche
                tile_img = img[y1:y2, x1:x2]
                
                tile_bboxes = []
                for bbox in bboxes:
                    cls, x_cen, y_cen, w_box, h_box = bbox
                    
                    # Convertir coordenadas normalizadas originales a píxeles absolutos de la imagen 4K
                    abs_x_cen = x_cen * w
                    abs_y_cen = y_cen * h
                    abs_w = w_box * w
                    abs_h = h_box * h
                    
                    # Calcular extremos de la caja original
                    box_x1 = abs_x_cen - (abs_w / 2)
                    box_y1 = abs_y_cen - (abs_h / 2)
                    box_x2 = abs_x_cen + (abs_w / 2)
                    box_y2 = abs_y_cen + (abs_h / 2)
                    
                    # Comprobar si el centro del defecto cae dentro del parche actual
                    if x1 <= abs_x_cen <= x2 and y1 <= abs_y_cen <= y2:
                        # Recortar la caja si sobresale del parche actual
                        new_x1 = max(box_x1, x1) - x1
                        new_y1 = max(box_y1, y1) - y1
                        new_x2 = min(box_x2, x2) - x1
                        new_y2 = min(box_y2, y2) - y1
                        
                        # Convertir a nuevo formato YOLO relativo al parche de 1280x1280
                        new_x_cen = ((new_x1 + new_x2) / 2) / tile_size
                        new_y_cen = ((new_y1 + new_y2) / 2) / tile_size
                        new_w = (new_x2 - new_x1) / tile_size
                        new_h = (new_y2 - new_y1) / tile_size
                        
                        tile_bboxes.append([cls, new_x_cen, new_y_cen, new_w, new_h])

                # 4. Guardar el parche y su etiqueta correspondiente
                tile_name = f"{img_path.stem}_tile_{tile_count}"
                cv2.imwrite(os.path.join(output_img_dir, f"{tile_name}.jpg"), tile_img)
                
                # El archivo se escribe SIEMPRE (si no hay bboxes, queda vacío como Background Image)
                with open(Path(output_label_dir) / f"{tile_name}.txt", 'w') as f:
                    for t_box in tile_bboxes:
                        f.write(f"{t_box[0]} {' '.join(map(str, t_box[1:]))}\n")
                        
                tile_count += 1

In [15]:
# Ejecutar Tiling para el set de ENTRENAMIENTO
tile_dataset(
    img_dir='deteccion_defectos-1/train/images',
    label_dir='deteccion_defectos-1/train/labels',
    output_img_dir='dataset_tiled/train/images',
    output_label_dir='dataset_tiled/train/labels',
    tile_size=1280,
    overlap=0.2 # 20% de solapamiento
)

# Ejecutar Tiling para el set de VALIDACIÓN
tile_dataset(
    img_dir='deteccion_defectos-1/valid/images',
    label_dir='deteccion_defectos-1/valid/labels',
    output_img_dir='dataset_tiled/valid/images',
    output_label_dir='dataset_tiled/valid/labels',
    tile_size=1280,
    overlap=0.2
)

Procesando Tiling: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 10.15it/s]


In [16]:
# Ejecutar Tiling para el set de TEST
tile_dataset(
    img_dir='deteccion_defectos-1/test/images',
    label_dir='deteccion_defectos-1/test/labels',
    output_img_dir='dataset_tiled/test/images',
    output_label_dir='dataset_tiled/test/labels',
    tile_size=1280,
    overlap=0.2
)

Procesando Tiling: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  9.46it/s]


In [18]:
import yaml
import os

# Define las rutas absolutas o relativas desde donde ejecutarás el entrenamiento
# Es mejor usar rutas completas para evitar errores de "File Not Found"
dataset_path = os.path.abspath("dataset_tiled")

data_config = {
    'path': dataset_path,      # Directorio raíz del dataset
    'train': 'train/images',   # Ruta relativa a 'path' para entrenamiento
    'val': 'valid/images',     # Ruta relativa a 'path' para validación
    'test': 'test/images',     # Ruta relativa a 'path' para pruebas (opcional)

    'nc': 2,                   # Número de clases
    'names': ['Corrido', 'Hueco'] # Asegúrate de que este orden sea el mismo de Roboflow
}

# Guardar el archivo
with open('data_tiled.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Archivo data_tiled.yaml creado con éxito.")

Archivo data_tiled.yaml creado con éxito.


In [ ]:
import os
# 1. ESTO SIEMPRE PRIMERO: Configuración estricta de memoria para la serie RTX 50
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from ultralytics import YOLO
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == '__main__':
    clear_gpu()

    ruta_mejor_modelo = r"C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\defectos_v11m\weights\best.pt"
    model = YOLO(ruta_mejor_modelo) 
    
    # Lanzamos el entrenamiento de ajuste fino de alta precisión
    model.train(
        data="data_tiled.yaml",    # Tu nuevo archivo que apunta a la carpeta 'dataset_tiled'
        epochs=180,                # Con 100 épocas de ajuste fino sobre el nuevo dataset es suficiente
        imgsz=1280,                # Mantenemos los 1280px para una coincidencia 1:1 con el tamaño del parche
        batch=2,                   # Conservamos batch=2 para asegurar que quepa holgadamente en tus 8GB de VRAM
        amp=True,                  # Precisión mixta activa para optimizar la velocidad y memoria
        workers=2,
        
        # --- Configuración Quirúrgica de Aprendizaje (Hiperparámetros) ---
        lr0=0.001,                 # Bajamos la tasa a la décima parte (por defecto es 0.01). Evita saltos bruscos.
        lrf=0.01,                  # Tasa de aprendizaje final ultra baja para congelar el modelo en su punto óptimo.
        warmup_epochs=0,           # Ponemos el calentamiento en 0. El modelo ya es maduro, no necesita empezar lento.
        
        # --- Maximizar Castigo al Error (Para romper el 90% de precisión) ---
        box=8.5,                   # Subimos a 8.5 para exigir máxima perfección geométrica al encuadrar el defecto.
        cls=1.8,                   # ¡LA PALANCA PRINCIPAL! Subimos a 1.8 para pulverizar los falsos positivos.
        
        # --- Aumentaciones Dinámicas Moderadas ---
        # Como tu dataset físico ya está pre-aumentado por el script de Tiling con Flips y R90,
        # bajamos la intensidad de las aumentaciones de YOLO para no deformar destructivamente el hilo.
        mosaic=0.5,                
        copy_paste=0.3,            
        mixup=0.10,
        
        # --- HARDWARE y REPOSITORIO ---
        cache=False,
        device=0,
        name='defectos_v11m_tiled_90plus', # Nombre de la nueva carpeta de salida
        verbose=False
    )

New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=8.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.8, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_tiled.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=180, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\defectos_v11m\weights\best.pt, momentum=0